In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, RFE
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

In [7]:
data_df = pd.read_csv("../processed_data/data_combined_training.csv", index_col=0)
data_df

,App A,App B,Num Nodes,App A Isolated Time,App B Isolated Time,App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=0.2_Mode=d],...,App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with App B,App B Co-Scheduled Time with App A
0,beatnik,fiesta,1,291.714286,166.879464,292.3125,168.25,292.687500,167.808036,295.000000,...,294.401786,165.808036,295.437500,170.633929,291.066964,166.290179,292.750000,166.236607,295.526786,168.491071
1,beatnik,lammps,1,291.714286,209.500000,292.3125,212.00,292.687500,213.000000,295.000000,...,294.401786,211.500000,295.437500,211.750000,291.066964,215.250000,292.750000,211.000000,292.651786,212.500000
2,beatnik,minife,1,291.714286,207.750000,292.3125,206.00,292.687500,208.250000,295.000000,...,294.401786,207.750000,295.437500,210.241071,291.066964,208.750000,292.750000,206.750000,295.250000,209.750000
3,beatnik,minivite,1,291.714286,259.000000,292.3125,257.25,292.687500,256.750000,295.000000,...,294.401786,257.500000,295.437500,258.250000,291.066964,258.500000,292.750000,258.250000,293.727679,261.250000
4,beatnik,tricount,1,291.714286,237.000000,292.3125,234.25,292.687500,235.500000,295.000000,...,294.401786,234.500000,295.437500,235.250000,291.066964,234.000000,292.750000,236.250000,294.218750,234.000000
5,fiesta,lammps,1,166.879464,209.500000,168.2500,212.00,167.808036,213.000000,166.973214,...,165.808036,211.500000,170.633929,211.750000,166.290179,215.250000,166.236607,211.000000,167.000000,212.250000
6,fiesta,minife,1,166.879464,207.750000,168.2500,206.00,167.808036,208.250000,166.973214,...,165.808036,207.750000,170.633929,210.241071,166.290179,208.750000,166.236607,206.750000,167.486607,208.750000
7,fiesta,minivite,1,166.879464,259.000000,168.2500,257.25,167.808036,256.750000,166.973214,...,165.808036,257.500000,170.633929,258.250000,166.290179,258.500000,166.236607,258.250000,167.584821,257.500000
8,fiesta,tricount,1,166.879464,237.000000,168.2500,234.25,167.808036,235.500000,166.973214,...,165.808036,234.500000,170.633929,235.250000,166.290179,234.000000,166.236607,236.250000,166.986607,235.000000
9,lammps,minife,1,209.500000,207.750000,212.0000,206.00,213.000000,208.250000,214.000000,...,211.500000,207.750000,211.750000,210.241071,215.250000,208.750000,211.000000,206.750000,212.750000,209.000000


In [8]:
# Identify column categories
app_a_isolated = 'App A Isolated Time'
app_b_isolated = 'App B Isolated Time'

inhib_a_cols = [c for c in data_df.columns if c.startswith('App A Co-Scheduled Time with Inhib')]
inhib_b_cols = [c for c in data_df.columns if c.startswith('App B Co-Scheduled Time with Inhib')]

cosched_a_col = 'App A Co-Scheduled Time with App B'
cosched_b_col = 'App B Co-Scheduled Time with App A'

# --- Row type 1: original row, drop "App B Co-Scheduled Time with App A" ---
data_df_type1 = data_df.drop(columns=[cosched_b_col]).copy()

# --- Row type 2: swap App A <-> App B, keep only "App A Co-Scheduled Time with App B" (post-swap) ---
data_df_type2 = data_df.copy()

# Swap App A / App B names
data_df_type2['App A'] = data_df['App B']
data_df_type2['App B'] = data_df['App A']

# Swap isolated times
data_df_type2[app_a_isolated] = data_df[app_b_isolated]
data_df_type2[app_b_isolated] = data_df[app_a_isolated]

# Swap inhibitor columns pairwise
# inhib_a_cols[i] pairs with inhib_b_cols[i] (same param suffix)
for col_a, col_b in zip(inhib_a_cols, inhib_b_cols):
    data_df_type2[col_a] = data_df[col_b]
    data_df_type2[col_b] = data_df[col_a]

# After swap, "App A Co-Scheduled Time with App B" should hold what was cosched_b_col
data_df_type2[cosched_a_col] = data_df[cosched_b_col]

# Drop the cosched_b column (only keep cosched_a)
data_df_type2 = data_df_type2.drop(columns=[cosched_b_col])

# --- Combine ---
data_df_extended = pd.concat([data_df_type1, data_df_type2], ignore_index=True)
data_df_extended

,App A,App B,Num Nodes,App A Isolated Time,App B Isolated Time,App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=0.2_Mode=d],...,App B Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with App B
0,beatnik,fiesta,1,291.714286,166.879464,292.3125,168.2500,292.687500,167.808036,295.000000,...,166.638393,294.401786,165.808036,295.437500,170.633929,291.066964,166.290179,292.750000,166.236607,295.526786
1,beatnik,lammps,1,291.714286,209.500000,292.3125,212.0000,292.687500,213.000000,295.000000,...,214.250000,294.401786,211.500000,295.437500,211.750000,291.066964,215.250000,292.750000,211.000000,292.651786
2,beatnik,minife,1,291.714286,207.750000,292.3125,206.0000,292.687500,208.250000,295.000000,...,207.500000,294.401786,207.750000,295.437500,210.241071,291.066964,208.750000,292.750000,206.750000,295.250000
3,beatnik,minivite,1,291.714286,259.000000,292.3125,257.2500,292.687500,256.750000,295.000000,...,263.000000,294.401786,257.500000,295.437500,258.250000,291.066964,258.500000,292.750000,258.250000,293.727679
4,beatnik,tricount,1,291.714286,237.000000,292.3125,234.2500,292.687500,235.500000,295.000000,...,234.500000,294.401786,234.500000,295.437500,235.250000,291.066964,234.000000,292.750000,236.250000,294.218750
5,fiesta,lammps,1,166.879464,209.500000,168.2500,212.0000,167.808036,213.000000,166.973214,...,214.250000,165.808036,211.500000,170.633929,211.750000,166.290179,215.250000,166.236607,211.000000,167.000000
6,fiesta,minife,1,166.879464,207.750000,168.2500,206.0000,167.808036,208.250000,166.973214,...,207.500000,165.808036,207.750000,170.633929,210.241071,166.290179,208.750000,166.236607,206.750000,167.486607
7,fiesta,minivite,1,166.879464,259.000000,168.2500,257.2500,167.808036,256.750000,166.973214,...,263.000000,165.808036,257.500000,170.633929,258.250000,166.290179,258.500000,166.236607,258.250000,167.584821
8,fiesta,tricount,1,166.879464,237.000000,168.2500,234.2500,167.808036,235.500000,166.973214,...,234.500000,165.808036,234.500000,170.633929,235.250000,166.290179,234.000000,166.236607,236.250000,166.986607
9,lammps,minife,1,209.500000,207.750000,212.0000,206.0000,213.000000,208.250000,214.000000,...,207.500000,211.500000,207.750000,211.750000,210.241071,215.250000,208.750000,211.000000,206.750000,212.750000


In [9]:
# ── 1. Separate features and label ──────────────────────────────────────────
label_col = "App A Co-Scheduled Time with App B"

X = data_df_extended.drop(columns=[label_col])
y = data_df_extended[label_col]

# Drop non-numeric or encode categoricals if App A / App B are strings
X = pd.get_dummies(X, columns=["App A", "App B"], drop_first=True)

# ── 2. Scale ─────────────────────────────────────────────────────────────────
scaler = StandardScaler()          # swap for MinMaxScaler() if you prefer [0,1]
X_scaled = scaler.fit_transform(X)
feature_names = X.columns.tolist()

K = 8   # max features to keep

# ── Option 1a: SelectKBest — F-regression (linear) ───────────────────────────
sel_f = SelectKBest(score_func=f_regression, k=K)
sel_f.fit(X_scaled, y)
features_f = [feature_names[i] for i in sel_f.get_support(indices=True)]
print("SelectKBest (F-regression):", features_f)

# ── Option 1b: SelectKBest — Mutual Information (nonlinear) ──────────────────
sel_mi = SelectKBest(score_func=mutual_info_regression, k=K)
sel_mi.fit(X_scaled, y)
features_mi = [feature_names[i] for i in sel_mi.get_support(indices=True)]
print("SelectKBest (Mutual Info): ", features_mi)

# ── Option 2: RFE with Ridge regression ──────────────────────────────────────
rfe = RFE(estimator=Ridge(), n_features_to_select=K, step=1)
rfe.fit(X_scaled, y)
features_rfe = [feature_names[i] for i in rfe.get_support(indices=True)]
print("RFE (Ridge):               ", features_rfe)

# ── Option 3: Random Forest importances ──────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)
importances = pd.Series(rf.feature_importances_, index=feature_names)
features_rf = importances.nlargest(K).index.tolist()
print("Random Forest importances: ", features_rf)

# ── Summary comparison ────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "F-regression":   pd.Series(1, index=features_f),
    "Mutual Info":    pd.Series(1, index=features_mi),
    "RFE (Ridge)":    pd.Series(1, index=features_rfe),
    "RandomForest":   pd.Series(1, index=features_rf),
}).fillna(0).astype(int)

print("\nFeature agreement across methods:")
print(summary)
print("\nFeatures selected by all methods:")
print(summary[summary.sum(axis=1) == 4].index.tolist())

SelectKBest (F-regression): ['App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=0.2_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=1.0_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=1.0_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.2_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d]']
SelectKBest (Mutual Info):  ['App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d]', 'App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=0.2_Mode=d]', 'App A Co

In [10]:
summary

,F-regression,Mutual Info,RFE (Ridge),RandomForest
App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],1,1,1,0
App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=1.0_Mode=d],1,1,1,1
App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.2_Mode=d],1,1,1,1
App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],1,1,1,1
App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=0.2_Mode=d],1,1,1,1
App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=0us_Sparsity=1.0_Mode=d],1,1,1,1
App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=0.2_Mode=d],1,1,1,1
App A Co-Scheduled Time with Inhib [MsgSz=35000_Wait=1000000us_Sparsity=1.0_Mode=d],1,1,1,1
App A Isolated Time,0,0,0,1
